RAG

chunck -> a small piece of a document

embedding -> A vector (list of numbers) representing text meaning

vector database -> a database that scores and searches vectors by similarity

retrieval -> finding the most relevant chunks for a given query

context injection -> Adding retrive chunks into the llm's prompt

grounding -> forcing the llm to answer based on the provided facts

knowledge base -> the collection of documents the system can retrieve from

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

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
print("All libraries imported successfully.")
print("Ready to build a RAG system")

All libraries imported successfully.
Ready to build a RAG system


In [ ]:
import os

#GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API clinet initialized.")
print("Note: If you see an authentication error later, double-check your api key.")

Groq API clinet initialized.
Note: If you see an authentication error later, double-check your api key.


In [ ]:
df=pd.read_csv('college_notes.csv')

print("Shape of dataset:", df.shape)

print("\nColumn names:", df.columns.tolist())

Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']


In [ ]:
df.head(5)

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
print("Subjects in the dataset:")

df['subject'].value_counts()

Subjects in the dataset:


,count
subject,
Data Engineering,5
Machine Learning,5
Generative AI,3
Python Programming,2


In [ ]:
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))


Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python Programming                 Pandas Library
   N015 Python Programming             Data Visualization

Length of content (number of characters) for each no

In [ ]:
documents = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas = [{'subject': row['subject'], 'topic': row['topic']} for row in df.to_dict('records')]

print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID: {ids[0]}")
print(f"First document metadata: {metadatas[0]}")
print(f"First document content: {documents[0][:100]}...")

Total chunks prepared: 15
First document ID: note_N001
First document metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("Loading embedding model...")
print("(This is take 30-60 seconds on first run-model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("\nEmbedding model loaded successfully.")
test_embedding = embedding_model.encode("This is a test sentence.")

print(f"Test embedding shape: {test_embedding.shape}")


Loading embedding model...
(This is take 30-60 seconds on first run-model is being downloaded)
(Subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Test embedding shape: (384,)


In [ ]:
print(f"First 5 values of test embedding: {test_embedding[:5]}")

First 5 values of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="college_notes.rag")

print("Chromadb client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

Chromadb client created.
Collection name: college_notes_rag
Documents in collection so far: 0


In [ ]:
print("Generating embeddings for all 15 notes")
print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)
print("\nEmbedding matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)

print(f"\nDocuments successfully added to ChromaDB.")
print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: {embeddings.shape}

Documents successfully added to ChromaDB.
Total documents in collection: 15


In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results

print("Retrieval function defines successfully.")
print("Function: retrieve_relevant_chunks(question, top_k=3")


Retrieval function defines successfully.
Function: retrieve_relevant_chunks(question, top_k=3


In [ ]:
test_question = "What is ETL and how does it work in data engineering?"
print(f"Test question: {test_question}")
print("="*60)

results = retrieve_relevant_chunks(test_question, top_k=3)

print("\nTop 3 most relevant chunks in the collection:")
print("="*60)

for i, (doc, dist, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"\nResult {i+1}:")
    print(f"  Subject: {meta['subject']}")
    print(f"  Topic: {meta['topic']}")
    print(f"  Distance: {dist:.4f}")
    print(f"  Content: {doc[:100]}...")

Test question: What is ETL and how does it work in data engineering?

Top 3 most relevant chunks in the collection:

Result 1:
  Subject: Data Engineering
  Topic: ETL Pipelines
  Distance: 0.2269
  Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...

Result 2:
  Subject: Data Engineering
  Topic: APIs and Data Collection
  Distance: 1.0690
  Content: An API or Application Programming Interface allows two software applications to talk to each other. ...

Result 3:
  Subject: Python Programming
  Topic: Data Visualization
  Distance: 1.3375
  Content: Data visualization is the process of representing data as charts graphs and visual formats. Python l...


Context Injection - We take the retrieved documents and paste them into the LLM prompt

User:

Context:

---
[Retrieved Document 1]

---
[Retrieved Document 2]

---
[Retrieved Document 3]

---
Question: [User's Question]

Answer:

In [ ]:
#create the function to connect and print the result where the llm prompt asked
def build_context_from_results(results):
  #result of collection.query()
  context_parts=[] #to collect the formated chuncks

  #loop to retrieve the document and metadata
  for i, (doc,meta) in enumerate(zip(
      results['documents'][0], #text data
      results['metadatas'][0])): #metadata(subject and topic) of text data

      #formate the chunck data into this
      chunk_text=f"[Source {i+1} : {meta['subject']} - {meta['topic']}]\n{doc}"
      #append the chunck into created context_parts
      context_parts.append(chunk_text)

  #make the context_parts into str and return str in the function
  context_str="\n\n---\n\n".join(context_parts)
  return context_str
context = build_context_from_results(results)
print("Built context string from retrieved chunks:")
print("="*60)

print(context[:500]+"...")
print(f"\nTotal context length: {len(context)}characters")

Built context string from retrieved chunks:
[Source 1 : Data Engineering - ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

[Source 2 : Data Engineering - APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services lik...

Total context length: 859characters


In [ ]:
#build the RAG generation function
#This function sends the context +question to groq llm and returns the answer

#there is two prompt : system and user
def generate_rag_answer(question,context):
  system_prompt="""
  You are a helpful academic assistent for engineering students.
  You will be given context retrieved from a college knowldge base, and a student question.
  RULES:
  1.Answer only using the information provided in the context below.
  2.If the answer is not found in the context, say exactly:
  "I don't have the enough knowledge base answer to answer this question."
  3.Do not use your general training knowledge.
  4.keep answers clear, accurate, and beginner friendly.
  5.mention which source the information came from when possible."""

  #user prompt
  user_prompt = f"""context from knowledge Base:
{context}
---
Student's question : {question}
please answer the question using the context provided above."""


#call the groq api with the msg
  response=groq_client.chat.completions.create(
      #model : llm
      model="llama-3.1-8b-instant",
      #msg for groq
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      #allocate the temperature value and maximum token value
      temperature=0.1,
      max_tokens=500
  )
  #Extract the answer from api response object
  #response.choices : a list of response options (usually 1)
  #[0] : take the first ( and only ) choice
  #.message.content : the actual text of the response
  answer = response.choices[0].message.content
  return answer
print("RAG generation function defined")

RAG generation function defined


In [ ]:
#CELL 14

#This is a complete RAG pipeline: Question->Retrieve ->inject

def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retrieving relevant documents...")

  #Step 1: Retrieve - find relevant chunks from chromadb
  results = retrieve_relevant_chunks(question, top_k=top_k)
  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i, meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']} - {meta['topic']}")
    print("\nStep 2: Building context string...")

  context = build_context_from_results(results)

  if verbose:
     print(f"Context built ({len(context)} characters)")
     print("\nStep 3: Sending to llm for answer generation")

  answer = generate_rag_answer(question, context)

  if verbose:
     print("\n" + "=" * 60)
     print("ANSWER:")
     print("=" * 60)
     print(answer)
     print("=" * 60)

  return answer

print("Completion RAG pipeline function ready.")
print("Function: ask_college_assistant(question, top_k=3)")

Completion RAG pipeline function ready.
Function: ask_college_assistant(question, top_k=3)


In [ ]:
question_1 = "What is ETL and how does it work in data engineering?"
answer_1 = ask_college_assistant(question_1, top_k=3, verbose=True)

Question: What is ETL and how does it work in data engineering?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Data Engineering - ETL Pipelines
 2. Data Engineering - APIs and Data Collection
 3. Python Programming - Data Visualization

Step 2: Building context string...
Context built (859 characters)

Step 3: Sending to llm for answer generation

ANSWER:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources (Source 1: Data Engineering - ETL Pipelines).

Here's a step-by-step explanation of how ETL works in data engineering:

1. **Extract**: Raw data is collected from various sources using APIs (Source 2: Data Engineering - APIs and Data Collection). For example, weather data, stock prices, or social media feeds.
2. **Transform**: The collected raw data is then transformed into a clean and structured format. This step is crucial as it prepares the data for analysis and storage.
3. **Load**: The t

In [ ]:
question_2 = "How do embeddings help in building search system?"
answer_2 = ask_college_assistant(question_2, top_k=3, verbose=True)

Question: How do embeddings help in building search system?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Generative AI - Retrieval Augmented Generation
 2. Generative AI - Large Language Models
 3. Machine Learning - Feature Engineering

Step 2: Building context string...
Context built (913 characters)

Step 3: Sending to llm for answer generation

ANSWER:
Unfortunately, the context provided does not directly address how embeddings help in building a search system. However, I can try to provide an indirect answer based on the information given.

From [Source 1: Generative AI - Retrieval Augmented Generation], we know that RAG (Retrieval Augmented Generation) is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This suggests that the model is using some form of similarity measurement to retrieve relevant documents.

Embeddings are a way to rep

In [ ]:
question_3 = "What is the population of Tokya?"
print("Testing with an out-of-scope question:")
print(f"Question: {question_3}")
answer_3 = ask_college_assistant(question_3, top_k=3, verbose=True)

Testing with an out-of-scope question:
Question: What is the population of Tokya?
Question: What is the population of Tokya?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Generative AI - Large Language Models
 2. Python Programming - Pandas Library
 3. Data Engineering - Big Data and PySpark

Step 2: Building context string...
Context built (887 characters)

Step 3: Sending to llm for answer generation

ANSWER:
I don't have the enough knowledge base answer to answer this question.

The context provided does not contain any information about the population of Tokyo or any other geographical location. It covers topics such as Generative AI, Python programming with Pandas library, and Data Engineering with Big Data and PySpark.


#Comparison Table
|Feature     | Without Rag  | With Rag  |
|:----------|:--------------|:--------------|
|**Knowledge source**| LLM training data(fixed)         | Your custom documents(updatable)|
| **Hallucination risk** | High | Low |
| **Can use private data** | No | Yes |
| **Knowledge cutoff**  | Yes(training data)  | No(u add new docs anytime) |
|**Cites sources**  | No | Yes(you know which chunk was used) |
| **Cost**   | Cheaper(shorter prompts)  |  Slightly higher(longer prompts with context) |

In [ ]:
def retrieve_by_subject(question, subject_filter, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject": {"$eq": subject_filter}}
  )
  return results

print("Retrieving only from a GenAI subject:")
print("="*50)

# Corrected function call: pass "How do LLMs generate text?" as the question
# and "Generative AI" as the subject_filter.
filtered_results = retrieve_by_subject(question="How do LLMs generate text?", subject_filter="Generative AI", top_k=2)
for i, (doc, meta) in enumerate(zip(filtered_results['documents'][0], filtered_results['metadatas'][0])):
  print(f"\nResult {i+1}: [{meta['subject']}]{meta['topic']}")
  print(f"  Content: {doc[:100]}...")

Retrieving only from a GenAI subject:

Result 1: [Generative AI]Large Language Models
  Content: A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generat...

Result 2: [Generative AI]Retrieval Augmented Generation
  Content: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant docu...


Beginner Questions

Q1. What is hallucination in the context of LLMs?

Q2. What does RAG stand for? What problem does it solve?

Q3. What us the role of a vector databse in the RAG pipeline?

Intermediate Questions

Q4. What is the difference between the indexing phase and the Querying phase of RAG?

Q5. Why must you use the same embedding model for both documents and queries?

Q6. Why is low temperature (e.g.0.1) preferred for RAG-based LLM calls?

CODING QUESTIONS:

Q7. Modify the ask_college_assistant() function to also display the distance scores of retrieved cunks in the output.

Q8. Change the system prompt in generate_rag_answer() to instruct the LLM to alwasys respond in bullet points

Q9. Add a function that returns only the topic names of retrieved chunks without their full content.

# Beginner Questions

### Q1. What is hallucination in the context of LLMs?

Hallucination occurs when an LLM generates information that appears correct but is actually false, inaccurate, or unsupported by facts.

### Q2. What does RAG stand for? What problem does it solve?

RAG stands for Retrieval-Augmented Generation. It solves the problem of outdated knowledge and hallucinations by retrieving relevant information from external documents before generating an answer.

### Q3. What is the role of a vector database in the RAG pipeline?

A vector database stores document embeddings and performs similarity search to retrieve the most relevant documents for a user's query.

---

# Intermediate Questions

### Q4. What is the difference between the indexing phase and querying phase?

Indexing Phase:
Documents are processed, converted into embeddings, and stored in the vector database.

Querying Phase:
The user's question is converted into an embedding, relevant documents are retrieved, and the LLM generates an answer using the retrieved context.

### Q5. Why must you use the same embedding model for both documents and queries?

Using the same embedding model ensures that documents and queries are represented in the same vector space, allowing accurate similarity comparisons.

### Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?

A low temperature reduces randomness and helps the model generate more factual, consistent, and context-grounded answers based on the retrieved documents.

---

# Coding Questions

### Q7. Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output.

Retrieve the distance scores from the vector database results and display them along with each retrieved chunk to indicate how closely the chunk matches the user's query.

### Q8. Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points.

Update the system prompt with an instruction such as:
"Always provide answers in bullet-point format."

### Q9. Add a function that returns only the topic names of retrieved chunks without their full content.

Create a function that retrieves relevant chunks and returns only the topic names from the metadata instead of returning the full document content.

# MINI PROJECT: College Knowledge Assistent

Project Description

Build a complete College Knowledge Assistant that:

1. Loads the college_notes.csv knowledge base

2. Indexes all notes in ChromaDB with embeddings

3. Accepts a student question

4. Retrieve the top 3 relevant notes

5. Injects them as context into a Groq LLm prompt

6. Returns a clear, grounded answer with source citations

7. Handles questions outside the knowledge base gracefully.

In [ ]:
# ==========================================================
# MINI PROJECT : COLLEGE KNOWLEDGE ASSISTANT
# ==========================================================

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# ----------------------------------------------------------
# 1. Load Knowledge Base
# ----------------------------------------------------------

df = pd.read_csv("college_notes.csv")

print(f"Loaded {len(df)} notes")

# ----------------------------------------------------------
# 2. Initialize Embedding Model
# ----------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------------------------------------
# 3. Create ChromaDB Collection
# ----------------------------------------------------------

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="college_notes"
)

# ----------------------------------------------------------
# 4. Store Notes with Embeddings
# ----------------------------------------------------------

documents = []
metadatas = []
ids = []

for i, row in df.iterrows():

    text = str(row["content"])

    documents.append(text)

    metadatas.append({
        "subject": row["subject"],
        "topic": row["topic"]
    })

    ids.append(str(i))

embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("Knowledge Base Indexed Successfully!")

# ----------------------------------------------------------
# 5. Initialize Groq
# ----------------------------------------------------------

groq_client = Groq(
   # api_key="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"
)

# ----------------------------------------------------------
# 6. RAG Function
# ----------------------------------------------------------

def ask_college_assistant(question, top_k=3):

    # Retrieve Relevant Notes
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # Handle Out-of-Scope Questions
    if len(docs) == 0:
        return "Sorry, this information is not available in the college knowledge base."

    # Build Context
    context = "\n\n".join(docs)

    prompt = f"""
You are a College Knowledge Assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say:
'Sorry, this information is not available in the college knowledge base.'
"""

    # Generate Answer
    response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    answer = response.choices[0].message.content

    # Add Citations
    citations = "\n\nSources:\n"

    for meta in metas:
        citations += f"- {meta['subject']} : {meta['topic']}\n"

    return answer + citations

# ----------------------------------------------------------
# 7. Ask Question
# ----------------------------------------------------------

question = input("Ask a Question: ")

answer = ask_college_assistant(question)

print("\nAnswer:\n")
print(answer)

Loaded 15 notes


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge Base Indexed Successfully!
Ask a Question: What is ETL?

Answer:

ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.

Sources:
- Data Engineering : ETL Pipelines
- Data Engineering : APIs and Data Collection
- Generative AI : Retrieval Augmented Generation

